In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D12 — Our World in Data — Annual CO2 Emissions Time Series
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter

import csv
import hashlib
import json
import re

import pandas as pd

DOCUMENT_ID = "D12"
DOCUMENT_NAME = "Our World in Data — Annual CO2 emissions time series"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "Deterministic complete CSV-to-Markdown table conversion "
    "with physical CSV row provenance"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".csv"
EXPECTED_SOURCE_SHA256 = "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788"

EXPECTED_COLUMNS = [
    "Year",
    "Annual CO2 emissions"
]

EXPECTED_SOURCE_ROW_COUNT = 275
EXPECTED_MIN_YEAR = 1750
EXPECTED_MAX_YEAR = 2024

TARGET_YEARS = [
    1750, 1800, 1850, 1900, 1950,
    1960, 1970, 1980, 1990, 2000,
    2010, 2011, 2012, 2013, 2014,
    2015, 2016, 2017, 2018, 2019,
    2020, 2021, 2022, 2023, 2024
]

EXPECTED_RECORD_COUNT = len(TARGET_YEARS)

REFERENCE_CATEGORY = "Environmental time-series"
REFERENCE_TOPIC = "Annual CO2 emissions"
REFERENCE_DESCRIPTION = "Annual CO2 emissions"

EXPECTED_CATEGORY_COUNTS = {
    REFERENCE_CATEGORY: EXPECTED_RECORD_COUNT
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Reporting Period",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

OUTPUT_DIR = Path("outputs_D12_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DIAGNOSTICS_PATH = OUTPUT_DIR / "D12_branch_B_source_diagnostics.json"
CONVERSION_INTEGRITY_PATH = OUTPUT_DIR / "D12_branch_B_conversion_integrity.json"
REPRESENTATION_PATH = OUTPUT_DIR / "D12_branch_B_representation.json"
STRUCTURAL_MARKDOWN_PATH = OUTPUT_DIR / "D12_branch_B_structural_markdown.md"
PROMPT_PATH = OUTPUT_DIR / "D12_branch_B_prompt.txt"
EXPERIMENT_METADATA_PRE_PATH = OUTPUT_DIR / "D12_branch_B_experiment_metadata_pre.json"

RAW_RESPONSE_PATH = OUTPUT_DIR / "D12_branch_B_raw_response.txt"
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D12_branch_B_parsed_extraction.json"
TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D12_branch_B_technical_diagnostics.json"
)
EXPERIMENT_METADATA_PATH = OUTPUT_DIR / "D12_branch_B_experiment_metadata.json"
EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / "D12_branch_B_experiment_summary.json"

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Frozen source SHA-256:", EXPECTED_SOURCE_SHA256)
print("Target years:", len(TARGET_YEARS))


In [ ]:
# ============================================================
# 1. Upload and verify the exact original D12 CSV
# ============================================================

uploaded = files.upload()

csv_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".csv")
]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one original D12 CSV."
    )

SOURCE_PATH = csv_paths[0]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def detect_delimiter(path):
    sample = path.read_text(
        encoding="utf-8-sig"
    )[:4096]

    try:
        dialect = csv.Sniffer().sniff(
            sample,
            delimiters=[",", ";", "\t", "|"]
        )
        return dialect.delimiter

    except csv.Error:
        raise ValueError(
            "Could not determine D12 CSV delimiter deterministically."
        )


SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D12 CSV does not match the frozen Stage 1 source identity."
    )

DETECTED_DELIMITER = detect_delimiter(
    SOURCE_PATH
)

source_df = pd.read_csv(
    SOURCE_PATH,
    sep=DETECTED_DELIMITER,
    encoding="utf-8-sig"
)

source_df.columns = [
    str(column).strip()
    for column in source_df.columns
]

OBSERVED_COLUMNS = (
    source_df.columns.tolist()
)

COLUMNS_VALID = (
    OBSERVED_COLUMNS
    == EXPECTED_COLUMNS
)

if not COLUMNS_VALID:
    raise ValueError(
        "D12 source columns do not match the frozen Stage 1 schema."
    )

# Parse copies for integrity checks only.
parsed_year = pd.to_numeric(
    source_df["Year"],
    errors="coerce"
)

parsed_value = pd.to_numeric(
    source_df["Annual CO2 emissions"],
    errors="coerce"
)

if parsed_year.isna().any():
    raise ValueError(
        "One or more D12 Year values are not numeric."
    )

if parsed_value.isna().any():
    raise ValueError(
        "One or more D12 emissions values are not numeric."
    )

if not all(float(value).is_integer() for value in parsed_year):
    raise ValueError(
        "D12 Year contains a non-integer source value."
    )

if not all(float(value).is_integer() for value in parsed_value):
    raise ValueError(
        "D12 Annual CO2 emissions contains a non-integer source value."
    )

SOURCE_YEARS = parsed_year.astype(int)
SOURCE_VALUES = parsed_value.astype(int)

SOURCE_ROW_COUNT = len(source_df)
SOURCE_COLUMN_COUNT = source_df.shape[1]

MIN_YEAR = int(SOURCE_YEARS.min())
MAX_YEAR = int(SOURCE_YEARS.max())

DUPLICATE_YEAR_COUNT = int(
    SOURCE_YEARS.duplicated().sum()
)

EXPECTED_YEAR_SEQUENCE = list(
    range(
        EXPECTED_MIN_YEAR,
        EXPECTED_MAX_YEAR + 1
    )
)

SOURCE_YEAR_SEQUENCE = (
    SOURCE_YEARS.tolist()
)

MISSING_SOURCE_YEARS = sorted(
    set(EXPECTED_YEAR_SEQUENCE)
    - set(SOURCE_YEAR_SEQUENCE)
)

UNEXPECTED_SOURCE_YEARS = sorted(
    set(SOURCE_YEAR_SEQUENCE)
    - set(EXPECTED_YEAR_SEQUENCE)
)

NULL_CELL_COUNT = int(
    source_df.isna().sum().sum()
)

SOURCE_INTEGRITY_VALID = all([
    SOURCE_HASH_MATCH,
    COLUMNS_VALID,
    SOURCE_ROW_COUNT == EXPECTED_SOURCE_ROW_COUNT,
    SOURCE_COLUMN_COUNT == len(EXPECTED_COLUMNS),
    MIN_YEAR == EXPECTED_MIN_YEAR,
    MAX_YEAR == EXPECTED_MAX_YEAR,
    DUPLICATE_YEAR_COUNT == 0,
    len(MISSING_SOURCE_YEARS) == 0,
    len(UNEXPECTED_SOURCE_YEARS) == 0,
    NULL_CELL_COUNT == 0,
    SOURCE_YEAR_SEQUENCE == EXPECTED_YEAR_SEQUENCE
])

SOURCE_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "detected_delimiter": DETECTED_DELIMITER,
    "observed_columns": OBSERVED_COLUMNS,
    "columns_valid": COLUMNS_VALID,
    "source_row_count": SOURCE_ROW_COUNT,
    "source_column_count": SOURCE_COLUMN_COUNT,
    "minimum_year": MIN_YEAR,
    "maximum_year": MAX_YEAR,
    "duplicate_year_count": DUPLICATE_YEAR_COUNT,
    "missing_source_years": MISSING_SOURCE_YEARS,
    "unexpected_source_years": UNEXPECTED_SOURCE_YEARS,
    "null_cell_count": NULL_CELL_COUNT,
    "source_integrity_valid": SOURCE_INTEGRITY_VALID,
    "machine_readable": True,
    "ocr_required": False
}

SOURCE_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        SOURCE_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

if not SOURCE_INTEGRITY_VALID:
    raise ValueError(
        "D12 frozen source integrity verification failed."
    )


In [ ]:
# ============================================================
# 2. Convert the COMPLETE CSV to structural Markdown
# ============================================================

def markdown_escape(value):
    text = str(value)

    return (
        text
        .replace("\\", "\\\\")
        .replace("|", "\\|")
        .replace("\r", " ")
        .replace("\n", " ")
    )


markdown_lines = [
    "# D12 — Our World in Data: Annual CO2 emissions time series",
    "",
    "> Complete structural conversion of the original CSV.",
    "> Every source data row is retained in original order.",
    "> `CSV data row` is deterministic provenance metadata derived from physical row position.",
    "",
    "| CSV data row | Year | Annual CO2 emissions |",
    "|---:|---:|---:|"
]

for dataframe_index in range(SOURCE_ROW_COUNT):

    physical_csv_row = (
        dataframe_index + 2
    )

    year = int(
        SOURCE_YEARS.iloc[
            dataframe_index
        ]
    )

    value = int(
        SOURCE_VALUES.iloc[
            dataframe_index
        ]
    )

    markdown_lines.append(
        "| "
        + str(physical_csv_row)
        + " | "
        + markdown_escape(year)
        + " | "
        + markdown_escape(value)
        + " |"
    )


STRUCTURAL_MARKDOWN_TEXT = (
    "\n".join(markdown_lines)
    + "\n"
)

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN_TEXT,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(
    STRUCTURAL_MARKDOWN_PATH
)

print(
    "Structural Markdown:",
    STRUCTURAL_MARKDOWN_PATH
)

print(
    "Representation SHA-256:",
    STRUCTURAL_MARKDOWN_SHA256
)

print(
    "Characters:",
    len(STRUCTURAL_MARKDOWN_TEXT)
)


In [ ]:
# ============================================================
# 3. Conversion-integrity verification
# ============================================================

table_row_pattern = re.compile(
    r"^\|\s*(\d+)\s*\|\s*(\d{4})\s*\|\s*(-?\d+)\s*\|$"
)

converted_rows = []

for line in STRUCTURAL_MARKDOWN_TEXT.splitlines():

    match = table_row_pattern.fullmatch(
        line.strip()
    )

    if match:

        converted_rows.append({
            "CSV data row":
                int(match.group(1)),

            "Year":
                int(match.group(2)),

            "Annual CO2 emissions":
                int(match.group(3))
        })


converted_df = pd.DataFrame(
    converted_rows
)

EXPECTED_PHYSICAL_ROWS = list(
    range(
        2,
        SOURCE_ROW_COUNT + 2
    )
)

conversion_row_count_valid = (
    len(converted_df)
    == SOURCE_ROW_COUNT
)

conversion_year_sequence_valid = (
    converted_df["Year"].tolist()
    == SOURCE_YEARS.tolist()
)

conversion_values_valid = (
    converted_df[
        "Annual CO2 emissions"
    ].tolist()
    == SOURCE_VALUES.tolist()
)

conversion_source_rows_valid = (
    converted_df[
        "CSV data row"
    ].tolist()
    == EXPECTED_PHYSICAL_ROWS
)

converted_years_unique = (
    converted_df["Year"].is_unique
)

converted_target_years_present = all(
    year in set(
        converted_df["Year"]
    )
    for year in TARGET_YEARS
)

source_and_conversion_pairs_identical = all(
    (
        int(converted_df.iloc[index]["Year"])
        == int(SOURCE_YEARS.iloc[index])
        and
        int(
            converted_df.iloc[index][
                "Annual CO2 emissions"
            ]
        )
        == int(SOURCE_VALUES.iloc[index])
    )
    for index in range(
        SOURCE_ROW_COUNT
    )
)

CONVERSION_INTEGRITY_PASSED = all([
    SOURCE_INTEGRITY_VALID,
    conversion_row_count_valid,
    conversion_year_sequence_valid,
    conversion_values_valid,
    conversion_source_rows_valid,
    converted_years_unique,
    converted_target_years_present,
    source_and_conversion_pairs_identical
])

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "expected_source_rows": SOURCE_ROW_COUNT,
    "converted_table_rows": int(len(converted_df)),
    "conversion_row_count_valid": conversion_row_count_valid,
    "conversion_year_sequence_valid": conversion_year_sequence_valid,
    "conversion_values_valid": conversion_values_valid,
    "conversion_source_rows_valid": conversion_source_rows_valid,
    "converted_years_unique": converted_years_unique,
    "all_target_years_present_in_complete_representation":
        converted_target_years_present,
    "source_and_conversion_pairs_identical":
        source_and_conversion_pairs_identical,
    "conversion_method":
        CONVERSION_METHOD,
    "complete_source_document_retained": True,
    "source_rows_removed": 0,
    "scope_filtering_applied": False,
    "target_year_filtering_applied": False,
    "row_reordering_applied": False,
    "ocr_applied": False,
    "semantic_rewriting_applied": False,
    "unit_inference_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "numeric_rescaling_applied": False,
    "numeric_rounding_applied": False,
    "manual_correction_applied": False,
    "normalisation_applied": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED
}

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)

if not CONVERSION_INTEGRITY_PASSED:
    raise ValueError(
        "D12 Branch B structural conversion failed integrity checks."
    )


In [ ]:
# ============================================================
# 4. Preserve Branch B representation metadata
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Complete structural Markdown table",

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "branch_name":
        BRANCH_NAME,

    "conversion_method":
        CONVERSION_METHOD,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "source_rows":
        SOURCE_ROW_COUNT,

    "converted_rows":
        len(converted_df),

    "structural_conversion_applied":
        True,

    "physical_csv_row_provenance_added":
        True,

    "complete_source_document_retained":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "target_year_filtering_applied":
        False,

    "ocr_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_correction_applied":
        False,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED
}

REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)


## Controlled Branch B extraction

The following task is the final D12 Branch A prompt adapted only to the Branch B representation and branch identifier.

The target-year list is part of the **fixed extraction task** and therefore remains identical across branches.

The model is **not supplied** with:
- Stage 1 reference values;
- expected source-row answers;
- Branch A extraction or Validation A results;
- a separate expected record-count number beyond the explicitly enumerated fixed target-year task.

The complete 275-row converted time series is the only source supplied to the model.

In [ ]:
# ============================================================
# 5. Create controlled Branch B extraction prompt
# ============================================================

TARGET_YEAR_TEXT = "\n".join(
    f"- {year}"
    for year in TARGET_YEARS
)

BRANCH_B_PROMPT = f"""You are an information extraction assistant.

Extract the predefined annual CO2-emissions observations represented
in the attached structural Markdown representation of the original CSV.

Treat the attached structural Markdown representation as the only
source of information.

The representation contains the complete annual time series with:

- CSV data row
- Year
- Annual CO2 emissions

The "CSV data row" column is structural provenance metadata identifying
the physical row of the original CSV, where the original header is
physical CSV row 1.

Extract one record for every year in the predefined target-year list
below.

Target years:

{TARGET_YEAR_TEXT}


For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location


Use these fixed semantic-field values for every extracted record:

Category:
Environmental time-series

Topic:
Annual CO2 emissions

Description:
Annual CO2 emissions

Unit:
null


Field rules:

Category:
- Use exactly:
  "Environmental time-series"

Topic:
- Use exactly:
  "Annual CO2 emissions"

Description:
- Use exactly:
  "Annual CO2 emissions"

Value:
- Extract the value represented in the "Annual CO2 emissions"
  column for the corresponding target year.
- Preserve the source numerical value.
- Return the value as a JSON number.
- Do not calculate, interpolate, estimate, rescale, round or convert
  the source value.
- Do not use values from neighbouring years.

Unit:
- Return null.
- The supplied source does not explicitly represent a separate
  measurement-unit field.
- Do not infer or introduce a unit from external knowledge.

Reporting Period:
- Use the corresponding target year as a string.
- Example format:
  "1750"

Source Location:
- Use the explicit "CSV data row" value associated with the target year
  in the structural Markdown.
- Return exactly this format:
  "CSV data row N"
- Do not infer row numbers from the target-year-list position.


Extraction rules:

- Extract only the predefined target years.
- Return one record for every year in the target-year list.
- Do not omit a listed year.
- Do not return years outside the predefined list.
- Use only values explicitly represented in the structural Markdown.
- Preserve the direct year-to-value association.
- Do not calculate missing values.
- Do not interpolate between years.
- Do not aggregate years.
- Do not introduce measurement units that are absent from the source.
- Do not use external knowledge.
- Do not follow external links.
- Do not modify or normalise source numerical values.
- Verify that every listed target year has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{{
  "document_id": "D12",
  "branch": "B",
  "records": [
    {{
      "Category": "Environmental time-series",
      "Topic": "Annual CO2 emissions",
      "Description": "Annual CO2 emissions",
      "Value": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }}
  ]
}}

Return only the JSON object.
"""

PROMPT_PATH.write_text(
    BRANCH_B_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


In [ ]:
# ============================================================
# 6. Pre-extraction experiment metadata
# ============================================================

EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_INTEGRITY_VALID,

    "detected_delimiter":
        DETECTED_DELIMITER,

    "source_rows":
        SOURCE_ROW_COUNT,

    "source_columns":
        SOURCE_COLUMN_COUNT,

    "source_year_range":
        f"{MIN_YEAR}-{MAX_YEAR}",

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "complete_source_document_retained":
        True,

    "scope_filtering_applied":
        False,

    "target_year_filtering_applied":
        False,

    "physical_csv_row_provenance_added":
        True,

    "ocr_applied_for_model_input":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_correction_applied":
        False,

    "fixed_extraction_task": {
        "target_years":
            TARGET_YEARS,
        "expected_fields":
            EXPECTED_FIELDS
    },

    "post_extraction_reference_diagnostics": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,
        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS
    },

    "reference_values_disclosed_to_model":
        False,

    "source_diagnostics_file":
        SOURCE_DIAGNOSTICS_PATH.name,

    "conversion_integrity_file":
        CONVERSION_INTEGRITY_PATH.name,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        "JSON object with document_id, branch and records",

    "execution_environment":
        "Independent ChatGPT conversation",

    "content_validation_performed":
        False,
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 7. Download Branch B model-input artefacts
# ============================================================

for path in [
    SOURCE_DIAGNOSTICS_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D12_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D12_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original CSV, Stage 1 reference dataset, "
    "Branch A outputs, or Validation A outputs.\n"
    "5. Save the first complete response exactly as returned as TXT.\n"
    "6. Do not correct, repair, reorder, deduplicate or regenerate it."
)


In [ ]:
# ============================================================
# 8. Upload and preserve the untouched model response
# ============================================================

uploaded_output = files.upload()

txt_paths = [
    Path(name)
    for name in uploaded_output
    if name.lower().endswith(".txt")
]

if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT file containing the complete D12 Branch B response."
    )

UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]

RAW_RESPONSE_TEXT = (
    UPLOADED_RAW_RESPONSE_PATH
    .read_text(
        encoding="utf-8"
    )
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError(
        "The uploaded D12 Branch B response is empty."
    )

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)

print(
    "Raw response preserved:",
    RAW_RESPONSE_PATH.name
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


In [ ]:
# ============================================================
# 9. Parse without repairing the response
# ============================================================

valid_json = False
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )
    valid_json = True

except json.JSONDecodeError as error:
    json_parsing_error = str(error)


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)

print("Valid JSON:", valid_json)
print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Records evaluable:", records_evaluable)
print("Observed record count:", observed_record_count)


In [ ]:
# ============================================================
# 10. Schema, field-type and D12 scope diagnostics
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(record, dict):
            record_structure_issues.append({
                "record_index":
                    record_index,
                "issue":
                    "Record is not a JSON object"
            })
            continue

        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index":
                    record_index,
                "issue":
                    "Field names or field order differ",
                "expected_fields":
                    EXPECTED_FIELDS,
                "observed_fields":
                    observed_fields
            })

        for field in STRING_OR_NULL_FIELDS:

            value = record.get(field)

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):
                field_type_issues.append({
                    "record_index":
                        record_index,
                    "field":
                        field,
                    "observed_type":
                        type(value).__name__,
                    "expected_type":
                        "string or null"
                })

        value = record.get("Value")

        if (
            isinstance(value, bool)
            or not isinstance(
                value,
                (int, float)
            )
        ):
            field_type_issues.append({
                "record_index":
                    record_index,
                "field":
                    "Value",
                "observed_type":
                    type(value).__name__,
                "expected_type":
                    "JSON number"
            })

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(field)

            if (
                value is None
                or (
                    isinstance(value, str)
                    and not value.strip()
                )
            ):
                missing_mandatory_values.append({
                    "record_index":
                        record_index,
                    "field":
                        field
                })


record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)


# ------------------------------------------------------------
# Content/scope diagnostics — separate from structure validity
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )

    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )

    observed_periods = [
        record.get(
            "Reporting Period"
        )
        for record in extracted_records
        if isinstance(
            record,
            dict
        )
    ]

    observed_period_counts = Counter(
        observed_periods
    )

    target_year_presence = {
        str(year):
            observed_period_counts[
                str(year)
            ]
            == 1
        for year in TARGET_YEARS
    }

    all_target_years_present_once = all(
        target_year_presence.values()
    )

    unexpected_periods = sorted(
        period
        for period
        in set(
            observed_periods
        )
        if period not in {
            str(year)
            for year in TARGET_YEARS
        }
    )

    no_unexpected_periods = (
        len(
            unexpected_periods
        )
        == 0
    )

    constant_fields_valid = all(
        (
            record.get("Category")
            == REFERENCE_CATEGORY
            and record.get("Topic")
            == REFERENCE_TOPIC
            and record.get("Description")
            == REFERENCE_DESCRIPTION
            and record.get("Unit")
            is None
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    source_location_pattern = re.compile(
        r"^CSV data row \d+$"
    )

    source_location_format_valid = all(
        (
            isinstance(
                record.get(
                    "Source Location"
                ),
                str
            )
            and source_location_pattern.fullmatch(
                record.get(
                    "Source Location"
                ).strip()
            )
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    duplicate_complete_record_count = sum(
        1
        for count
        in Counter(
            tuple(
                json.dumps(
                    record.get(field),
                    ensure_ascii=False,
                    sort_keys=True
                )
                for field
                in EXPECTED_FIELDS
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        ).values()
        if count > 1
    )

else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    observed_periods = None
    target_year_presence = None
    all_target_years_present_once = None
    unexpected_periods = None
    no_unexpected_periods = None
    constant_fields_valid = None
    source_location_format_valid = None
    duplicate_complete_record_count = None


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(
                missing_mandatory_values
            )
            if records_evaluable
            else None
        ),

    "all_target_years_present_once":
        all_target_years_present_once,

    "target_year_presence":
        target_year_presence,

    "unexpected_reporting_periods":
        unexpected_periods,

    "no_unexpected_reporting_periods":
        no_unexpected_periods,

    "constant_fields_valid":
        constant_fields_valid,

    "source_location_format_valid":
        source_location_format_valid,

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_count
}

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 11. Determine technical/schema validity
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "missing_mandatory_values":
        missing_mandatory_values
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

In [ ]:
# ============================================================
# 12. Preserve parsed extraction if structurally evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None

if structurally_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get("document_id"),

        "branch":
            parsed_response.get("branch"),

        "records":
            extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_created = True

    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:
    print(
        "No parsed extraction created because "
        "the output is not structurally evaluable."
    )

In [ ]:
# ============================================================
# 13. Final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "notes":
        (
            "D12 Branch B converts the complete 275-row source CSV "
            "to a deterministic Markdown table while preserving row "
            "order, years and emissions values exactly. Physical CSV "
            "row numbers are added as structural provenance metadata. "
            "The representation is not filtered to the target years. "
            "No OCR, semantic rewriting, unit inference, calculation, "
            "rescaling, rounding, normalisation or manual correction "
            "is applied. Stage 1 reference values are not disclosed to "
            "the model. Content-level validation is performed separately in "
            "Validation B — D12."
        )
}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_INTEGRITY_VALID,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY_PASSED,

    "structural_conversion_applied":
        True,

    "complete_source_document_retained":
        True,

    "target_year_filtering_applied":
        False,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "valid_json":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),


    "scope_complete":
        record_count_valid,

    "all_target_years_present_once":
        all_target_years_present_once,

    "no_unexpected_reporting_periods":
        no_unexpected_periods,

    "constant_fields_valid":
        constant_fields_valid,

    "source_location_format_valid":
        source_location_format_valid,

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_count,

    "parsed_extraction_created":
        bool(structurally_evaluable),

    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D12."
    )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 14. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    SOURCE_DIAGNOSTICS_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    GENERATED_OUTPUTS.append(
        PARSED_EXTRACTION_PATH
    )

print(
    "Generated D12 Branch B files:"
)

for path in GENERATED_OUTPUTS:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )

for path in GENERATED_OUTPUTS:
    files.download(path)
